## Hydrodynamic model of the Southern North Sea

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
#IMPORT LIBRARIES
import mikeio
import modelskill as ms
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import cmcrameri.cm as cmc
import pandas as pd

# Add parent directory to path to access src module
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent.parent))
print("Current working directory:", Path.cwd())
print("Parent directory added to path:", Path.cwd().parent.parent)

# dataset helper
from helpers import get_altimetry_obs, get_wl_point_obs
from helpers import plot_shred_reconstruction, extract_water_level_measurements, eval_all

import sys
from pathlib import Path

# Add parent directory to path to access src module
sys.path.insert(0, str(Path.cwd().parent.parent))

# Importing modules from src
from src.datamanager import DataManager
from src.engine import SHREDEngine
from src.train import fit, evaluate
from src.mSHRED.mSHRED import mSHRED
from src.utils import plot_training_curves, evaluate_model

# get device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Current working directory: /home/matteo/Documents/Masked-SHRED/Notebooks_final/HYDRO
Parent directory added to path: /home/matteo/Documents/Masked-SHRED


## Data Load

In [3]:
# folders
dfs_fldr = "datasets/output/Area.dfsu"
msh_fldr = "datasets/input/Mesh_8000_elems_HD_v2.mesh"
obs_fldr  = "datasets/observations/"

msh = mikeio.open(msh_fldr)
dfs = mikeio.open(dfs_fldr)
geom = dfs.geometry

# read datasets
ds = dfs.read(time = slice("2022-01-01", "2023-10-30"))
dsH = ds["Surface elevation"]
dsU = ds["U velocity"]
dsV = ds["V velocity"]

del dfs

# sensor coverage
alllist = get_wl_point_obs()

# For sparse data simulation
hf = ["Dover", "ZeebruggeWielingendok", "NWA", "K141"]
alllist = [s for s in alllist if s.name in hf]

stations = []
available_stations = []
intermittent = []
for i in range(len(alllist)):
    stations.append(alllist[i])
    if ds.time.isin(alllist[i].time).all():
        available_stations.append(alllist[i])
    else:
        intermittent.append(alllist[i])

# extract water level measurements for all stations
measurements = extract_water_level_measurements(stations, ds_time=ds.time, obs_fldr=obs_fldr)

fig, axs = plt.subplots(1,3, figsize = (8,3))
ms.plotting.spatial_overview(stations, mod = msh.geometry, figsize = (5,5), ax=axs[0], title=f"Stations: {len(stations)}")
ms.plotting.spatial_overview(available_stations, mod = msh.geometry, figsize = (5,5), ax=axs[1], title=f"Always available stations: {len(available_stations)}/{len(stations)}")
ms.plotting.spatial_overview(intermittent, mod = msh.geometry, figsize = (5,5), ax=axs[2], title=f"Intermittent stations: {len(intermittent)}/{len(stations)}")
plt.tight_layout()
plt.show()

# remove nans from H, U, V 
dsH = dsH.values[:,~np.isnan(ds["Surface elevation"].values).any(axis=0)]
dsU = dsU.values[:,~np.isnan(ds["Surface elevation"].values).any(axis=0)]
dsV = dsV.values[:,~np.isnan(ds["Surface elevation"].values).any(axis=0)]

# outputs
H = torch.tensor(dsH, dtype=torch.float32)
U = torch.tensor(dsU, dtype=torch.float32)
V = torch.tensor(dsV, dtype=torch.float32)

# inputs
measurements = torch.tensor(measurements, dtype=torch.float32)

# Extract measurements for always-available stations only
av_indices = [stations.index(s) for s in available_stations]
measurements_av = measurements[:, av_indices]

# print the shape of the arrays
print("H shape:", H.shape)
print("U shape:", U.shape)
print("V shape:", V.shape)
print("Measurements shape:", measurements.shape)
print("Always-available measurements shape:", measurements_av.shape)

# free memory
del dsH, dsU, dsV

FileNotFoundError: File datasets/input/Mesh_8000_elems_HD_v2.mesh not found

In [ ]:
ERRORS = []

# DataManager
LAGS = 48

TRAIN_SIZE = 0.8        # percentage of data used for training 
VAL_SIZE = 0.1          # percentage of data used for validation
TEST_SIZE = 0.1         # percentage of data used for testing

COMPRESS_H = 12         # modes kept for H (surface elevation)
COMPRESS_U = 40         # modes kept for U (U velocity)
COMPRESS_V = 68         # modes kept for V (V velocity)

# Training
OPTIMIZER = "AdamW"     # optimizer to use
LEARNING_RATE = 1e-3    # learning rate for training
NUM_EPOCHS = 200        # maximum number of epochs for training
LR_SCHEDULER = "CosineAnnealingLR"      # learning rate scheduler to use
SCHEDULER_STEP_SIZE = NUM_EPOCHS        # step size for learning rate scheduler (if applicable)

BATCH_SIZE = 32         # batch size for training
PATIENCE = 50           # number of epochs with no improvement on validation loss before stopping training
VERBOSE_TRAIN = True

# Architecture
ENCODER_DEPTH = 3              # number of encoder layers
HIDDEN_SIZE = 64                # hidden size for encoder and decoder layers
N_HEADS = 4                     # number of attention heads (if using a transformer-based architecture)
DIM_FEEDFORWARD = 96            # dimension of feedforward network in transformer (if using a transformer-based architecture)
DECODER = "cnn"                 # type of decoder to use (mpl or cnn)   
DECODER_HIDDEN_SIZES = [100]     # hidden sizes for decoder layers 
DROPOUT = 0.0                   # dropout rate for encoder and decoder layers
MASKING_TYPE = "mask_data"      # type of masking to apply to inputs during training (mask_data, mask_data_random, etc.)

## (T.0.A) Always available sensor

In [ ]:
always_available_sensors = available_stations

In [ ]:
# Data manager
manager = DataManager(lags=LAGS, train_size=TRAIN_SIZE, val_size=VAL_SIZE, test_size=TEST_SIZE)
# add data to manager
manager.add_data(H, id="Surface elevation", compress=COMPRESS_H, measurements=measurements_av)
manager.add_data(U, id="U velocity", compress=COMPRESS_U)
manager.add_data(V, id="V velocity", compress=COMPRESS_V)
# prepare datasets
train_dataset, val_dataset, test_dataset = manager.prepare()

# network input and output shapes
nn_input_shape = train_dataset[0][0].shape[1]
nn_output_shape = train_dataset[0][1].shape[0]
print("\nInput shape for the model:", nn_input_shape)
print("Output shape for the model:", nn_output_shape)



In [ ]:
# mSHRED TRAINING
model_args = dict(
    # Encoder
    encoder="transformer", encoder_depth=ENCODER_DEPTH, input_size=nn_input_shape, n_heads=N_HEADS,
    dim_feedforward=DIM_FEEDFORWARD, hidden_size=HIDDEN_SIZE, input_length=LAGS, double_channels=False,
    # Decoder
    decoder=DECODER, decoder_hidden_sizes=DECODER_HIDDEN_SIZES, output_size=nn_output_shape, dropout=DROPOUT,
)
mshred = mSHRED(args=model_args).to(device)

model_path = Path("models_saved/T0A.pth")
model_path.parent.mkdir(parents=True, exist_ok=True)

#if not already trained, train the model and save it, otherwise load the model from disk
if not model_path.exists():
    print("Training mSHRED model...")
    train_errors, valid_errors = fit(
        mshred, train_dataset, val_dataset,
        optimizer=OPTIMIZER, lr=LEARNING_RATE, num_epochs=NUM_EPOCHS, patience=PATIENCE, verbose=VERBOSE_TRAIN,
        lr_scheduler=LR_SCHEDULER, scheduler_step_size=SCHEDULER_STEP_SIZE,
    )
    torch.save(mshred.state_dict(), model_path)

else:
    print("Loading pre-trained mSHRED model...")
    mshred.load_state_dict(torch.load(model_path, map_location=device))
    mshred.eval()

eval_all(manager, mshred, train_dataset, val_dataset, test_dataset, H, U, V, model_name="T0A")

del model_args, model_path, mshred


## (T.0.B) Always available sensor (LSTM)


In [ ]:
# mSHRED TRAINING
model_args = dict(
    # Encoder
    encoder="lstm", input_size=nn_input_shape, hidden_size=HIDDEN_SIZE, num_layers=ENCODER_DEPTH,
    # Decoder
    decoder=DECODER, decoder_hidden_sizes=DECODER_HIDDEN_SIZES, output_size=nn_output_shape, dropout=DROPOUT,
)
mshred = mSHRED(args=model_args).to(device)

model_path = Path("models_saved/T0B.pth")
model_path.parent.mkdir(parents=True, exist_ok=True)
model_exists = model_path.exists()

#if not already trained, train the model and save it, otherwise load the model from disk
if not model_exists:
    print("Training mSHRED model...")

    # Use cosine annealing LR schedule
    train_errors, valid_errors = fit(
        mshred, train_dataset, val_dataset,
        optimizer=OPTIMIZER, lr=LEARNING_RATE, num_epochs=NUM_EPOCHS, patience=PATIENCE, verbose=VERBOSE_TRAIN,
        lr_scheduler=LR_SCHEDULER, scheduler_step_size=SCHEDULER_STEP_SIZE,
    )

    torch.save(mshred.state_dict(), model_path)

else:
    print("Loading pre-trained mSHRED model...")
    mshred.load_state_dict(torch.load(model_path, map_location=device))
    mshred.eval()

eval_all(manager, mshred, train_dataset, val_dataset, test_dataset, H, U, V, model_name="T0B")

del model_args, model_path, mshred
del model_exists
del train_dataset, val_dataset, test_dataset


## (T.1.A): Median imputer

In [ ]:
# Data manager
manager = DataManager(lags=LAGS, train_size=TRAIN_SIZE, val_size=VAL_SIZE, test_size=TEST_SIZE)
# add data to manager
manager.add_data(H, id="Surface elevation", compress=COMPRESS_H, measurements=measurements, imputer="median")
manager.add_data(U, id="U velocity", compress=COMPRESS_U)
manager.add_data(V, id="V velocity", compress=COMPRESS_V)
# prepare datasets
train_dataset, val_dataset, test_dataset = manager.prepare()

# network input and output shapes
nn_input_shape = train_dataset[0][0].shape[1]
nn_output_shape = train_dataset[0][1].shape[0]
print("\nInput shape for the model:", nn_input_shape)
print("Output shape for the model:", nn_output_shape)



In [ ]:
# mSHRED TRAINING
model_args = dict(
    # Encoder
    encoder="transformer", encoder_depth=ENCODER_DEPTH, input_size=nn_input_shape, n_heads=N_HEADS,
    dim_feedforward=DIM_FEEDFORWARD, hidden_size=HIDDEN_SIZE, input_length=LAGS, double_channels=False,
    # Decoder
    decoder=DECODER, decoder_hidden_sizes=DECODER_HIDDEN_SIZES, output_size=nn_output_shape, dropout=DROPOUT,
)
mshred = mSHRED(args=model_args).to(device)

model_path = Path("models_saved/T1A.pth")
model_path.parent.mkdir(parents=True, exist_ok=True)
model_exists = model_path.exists()

#if not already trained, train the model and save it, otherwise load the model from disk
if not model_exists:
    print("Training mSHRED model...")

    # Use cosine annealing LR schedule
    train_errors, valid_errors = fit(
        mshred, train_dataset, val_dataset,
        optimizer=OPTIMIZER, lr=LEARNING_RATE, num_epochs=NUM_EPOCHS, patience=PATIENCE, verbose=VERBOSE_TRAIN,
        lr_scheduler=LR_SCHEDULER, scheduler_step_size=SCHEDULER_STEP_SIZE,
    )

    torch.save(mshred.state_dict(), model_path)

else:
    print("Loading pre-trained mSHRED model...")
    mshred.load_state_dict(torch.load(model_path, map_location=device))
    mshred.eval()

eval_all(manager, mshred, train_dataset, val_dataset, test_dataset, H, U, V, model_name="T1A")

del model_args, model_path, mshred
del model_exists


## (T.1.B): Mean imputer

In [ ]:
# Data manager
manager = DataManager(lags=LAGS, train_size=TRAIN_SIZE, val_size=VAL_SIZE, test_size=TEST_SIZE)
# add data to manager
manager.add_data(H, id="Surface elevation", compress=COMPRESS_H, measurements=measurements, imputer="mean")
manager.add_data(U, id="U velocity", compress=COMPRESS_U)
manager.add_data(V, id="V velocity", compress=COMPRESS_V)
# prepare datasets
train_dataset, val_dataset, test_dataset = manager.prepare()

# network input and output shapes
nn_input_shape = train_dataset[0][0].shape[1]
nn_output_shape = train_dataset[0][1].shape[0]
print("\nInput shape for the model:", nn_input_shape)
print("Output shape for the model:", nn_output_shape)



In [ ]:
# mSHRED TRAINING
model_args = dict(
    # Encoder
    encoder="transformer", encoder_depth=ENCODER_DEPTH, input_size=nn_input_shape, n_heads=N_HEADS,
    dim_feedforward=DIM_FEEDFORWARD, hidden_size=HIDDEN_SIZE, input_length=LAGS, double_channels=False,
    # Decoder
    decoder=DECODER, decoder_hidden_sizes=DECODER_HIDDEN_SIZES, output_size=nn_output_shape, dropout=DROPOUT,
)
mshred = mSHRED(args=model_args).to(device)

model_path = Path("models_saved/T1B.pth")
model_path.parent.mkdir(parents=True, exist_ok=True)
model_exists = model_path.exists()

#if not already trained, train the model and save it, otherwise load the model from disk
if not model_exists:
    print("Training mSHRED model...")

    # Use cosine annealing LR schedule
    train_errors, valid_errors = fit(
        mshred, train_dataset, val_dataset,
        optimizer=OPTIMIZER, lr=LEARNING_RATE, num_epochs=NUM_EPOCHS, patience=PATIENCE, verbose=VERBOSE_TRAIN,
        lr_scheduler=LR_SCHEDULER, scheduler_step_size=SCHEDULER_STEP_SIZE,
    )

    torch.save(mshred.state_dict(), model_path)

else:
    print("Loading pre-trained mSHRED model...")
    mshred.load_state_dict(torch.load(model_path, map_location=device))
    mshred.eval()

eval_all(manager, mshred, train_dataset, val_dataset, test_dataset, H, U, V, model_name="T1B")

del model_args, model_path, mshred
del model_exists
del train_dataset, val_dataset, test_dataset


## (T.1.C): Last value imputer

In [ ]:
# Data manager
manager = DataManager(lags=LAGS, train_size=TRAIN_SIZE, val_size=VAL_SIZE, test_size=TEST_SIZE)
# add data to manager
manager.add_data(H, id="Surface elevation", compress=COMPRESS_H, measurements=measurements, imputer="median")
manager.add_data(U, id="U velocity", compress=COMPRESS_U)
manager.add_data(V, id="V velocity", compress=COMPRESS_V)
# prepare datasets
train_dataset, val_dataset, test_dataset = manager.prepare()

# network input and output shapes
nn_input_shape = train_dataset[0][0].shape[1]
nn_output_shape = train_dataset[0][1].shape[0]
print("\nInput shape for the model:", nn_input_shape)
print("Output shape for the model:", nn_output_shape)



In [ ]:
# mSHRED TRAINING
model_args = dict(
    # Encoder
    encoder="transformer", encoder_depth=ENCODER_DEPTH, input_size=nn_input_shape, n_heads=N_HEADS,
    dim_feedforward=DIM_FEEDFORWARD, hidden_size=HIDDEN_SIZE, input_length=LAGS, double_channels=False,
    # Decoder
    decoder=DECODER, decoder_hidden_sizes=DECODER_HIDDEN_SIZES, output_size=nn_output_shape, dropout=DROPOUT,
)
mshred = mSHRED(args=model_args).to(device)

model_path = Path("models_saved/T1C.pth")
model_path.parent.mkdir(parents=True, exist_ok=True)
model_exists = model_path.exists()

#if not already trained, train the model and save it, otherwise load the model from disk
if not model_exists:
    print("Training mSHRED model...")

    # Use cosine annealing LR schedule
    train_errors, valid_errors = fit(
        mshred, train_dataset, val_dataset,
        optimizer=OPTIMIZER, lr=LEARNING_RATE, num_epochs=NUM_EPOCHS, patience=PATIENCE, verbose=VERBOSE_TRAIN,
        lr_scheduler=LR_SCHEDULER, scheduler_step_size=SCHEDULER_STEP_SIZE,
    )

    torch.save(mshred.state_dict(), model_path)

else:
    print("Loading pre-trained mSHRED model...")
    mshred.load_state_dict(torch.load(model_path, map_location=device))
    mshred.eval()

eval_all(manager, mshred, train_dataset, val_dataset, test_dataset, H, U, V, model_name="T1C")

del model_args, model_path, mshred
del model_exists
del train_dataset, val_dataset, test_dataset


## (T.2.A) GRUD

In [ ]:
# Data manager
manager = DataManager(lags=LAGS, train_size=TRAIN_SIZE, val_size=VAL_SIZE, test_size=TEST_SIZE)
# add data to manager
manager.add_data(H, id="Surface elevation", compress=COMPRESS_H, measurements=measurements)
manager.add_data(U, id="U velocity", compress=COMPRESS_U)
manager.add_data(V, id="V velocity", compress=COMPRESS_V)
# prepare datasets
train_dataset, val_dataset, test_dataset, X_mean = manager.prepare(masking_type="grud")

# network input and output shapes
nn_input_shape = train_dataset[0][0].shape[-1]
nn_output_shape = train_dataset[0][1].shape[0]
print("\nInput shape for the model:", nn_input_shape)
print("Output shape for the model:", nn_output_shape)



In [ ]:
# GRUD TRAINING
model_args = dict(
    # Encoder
    encoder="grud", input_size=nn_input_shape, hidden_size=HIDDEN_SIZE, X_mean=X_mean,
    # Decoder
    decoder=DECODER, decoder_hidden_sizes=DECODER_HIDDEN_SIZES, output_size=nn_output_shape, dropout=DROPOUT,
)
mshred = mSHRED(args=model_args).to(device)

model_path = Path("models_saved/T2A.pth")
model_path.parent.mkdir(parents=True, exist_ok=True)
model_exists = model_path.exists()

#if not already trained, train the model and save it, otherwise load the model from disk
if not model_exists:
    print("Training mSHRED model...")

    # Use cosine annealing LR schedule
    train_errors, valid_errors = fit(
        mshred, train_dataset, val_dataset,
        optimizer=OPTIMIZER, lr=LEARNING_RATE, num_epochs=NUM_EPOCHS, patience=PATIENCE, verbose=VERBOSE_TRAIN,
        lr_scheduler=LR_SCHEDULER, scheduler_step_size=SCHEDULER_STEP_SIZE,
        masking_type="grud",
    )

    torch.save(mshred.state_dict(), model_path)

else:
    print("Loading pre-trained mSHRED model...")
    mshred.load_state_dict(torch.load(model_path, map_location=device))
    mshred.eval()

eval_all(manager, mshred, train_dataset, val_dataset, test_dataset, H, U, V, model_name="T2A")

del model_args, model_path, mshred
del model_exists


## (T.2.B): Attn-GRUD

In [ ]:
# Attn-GRUD TRAINING
model_args = dict(
    # Encoder
    encoder="att_grud", input_size=nn_input_shape, hidden_size=HIDDEN_SIZE, X_mean=X_mean,
    # Decoder
    decoder=DECODER, decoder_hidden_sizes=DECODER_HIDDEN_SIZES, output_size=nn_output_shape, dropout=DROPOUT,
)
mshred = mSHRED(args=model_args).to(device)

model_path = Path("models_saved/T2B.pth")
model_path.parent.mkdir(parents=True, exist_ok=True)
model_exists = model_path.exists()

#if not already trained, train the model and save it, otherwise load the model from disk
if not model_exists:
    print("Training mSHRED model...")

    # Use cosine annealing LR schedule
    train_errors, valid_errors = fit(
        mshred, train_dataset, val_dataset,
        optimizer=OPTIMIZER, lr=LEARNING_RATE, num_epochs=NUM_EPOCHS, patience=PATIENCE, verbose=VERBOSE_TRAIN,
        lr_scheduler=LR_SCHEDULER, scheduler_step_size=SCHEDULER_STEP_SIZE,
        masking_type="grud",
    )

    torch.save(mshred.state_dict(), model_path)

else:
    print("Loading pre-trained mSHRED model...")
    mshred.load_state_dict(torch.load(model_path, map_location=device))
    mshred.eval()

eval_all(manager, mshred, train_dataset, val_dataset, test_dataset, H, U, V, model_name="T2B")

del model_args, model_path, mshred
del model_exists
del train_dataset, val_dataset, test_dataset


## (T.3.A): Masking

In [ ]:
# Data manager
manager = DataManager(lags=LAGS, train_size=TRAIN_SIZE, val_size=VAL_SIZE, test_size=TEST_SIZE)
# add data to manager
manager.add_data(H, id="Surface elevation", compress=COMPRESS_H, measurements=measurements)
manager.add_data(U, id="U velocity", compress=COMPRESS_U)
manager.add_data(V, id="V velocity", compress=COMPRESS_V)
# prepare datasets
train_dataset, val_dataset, test_dataset = manager.prepare(masking_type=MASKING_TYPE)

# network input and output shapes
nn_input_shape = train_dataset.X.shape[-1]
nn_output_shape = train_dataset.Y.shape[-1]
print("\nInput shape for the model:", nn_input_shape)
print("Output shape for the model:", nn_output_shape)



In [ ]:
# mSHRED TRAINING
model_args = dict(
    # Encoder
    encoder="transformer", encoder_depth=ENCODER_DEPTH, input_size=nn_input_shape, n_heads=N_HEADS,
    dim_feedforward=DIM_FEEDFORWARD, hidden_size=HIDDEN_SIZE, input_length=LAGS, double_channels=False,
    # Decoder
    decoder=DECODER, decoder_hidden_sizes=DECODER_HIDDEN_SIZES, output_size=nn_output_shape, dropout=DROPOUT, 
)
mshred = mSHRED(args=model_args).to(device)

model_path = Path("models_saved/T3A.pth")
model_path.parent.mkdir(parents=True, exist_ok=True)
model_exists = model_path.exists()

#if not already trained, train the model and save it, otherwise load the model from disk
if not model_exists:
    print("Training mSHRED model...")

    # Use cosine annealing LR schedule
    train_errors, valid_errors = fit(
        mshred, train_dataset, val_dataset,
        optimizer=OPTIMIZER, lr=LEARNING_RATE, num_epochs=NUM_EPOCHS, patience=PATIENCE, verbose=VERBOSE_TRAIN,
        lr_scheduler=LR_SCHEDULER, scheduler_step_size=SCHEDULER_STEP_SIZE,
        masking_type=MASKING_TYPE,
    )

    torch.save(mshred.state_dict(), model_path)

else:
    print("Loading pre-trained mSHRED model...")
    mshred.load_state_dict(torch.load(model_path, map_location=device))
    mshred.eval()

eval_all(manager, mshred, train_dataset, val_dataset, test_dataset, H, U, V, model_name="T3A")

del model_args, model_path, mshred
del model_exists


## (T.3.B): Masking x2 channels

In [ ]:
# mSHRED TRAINING
model_args = dict(
    # Encoder
    encoder="transformer", encoder_depth=ENCODER_DEPTH, input_size=nn_input_shape, n_heads=N_HEADS,
    dim_feedforward=DIM_FEEDFORWARD, hidden_size=HIDDEN_SIZE, input_length=LAGS, double_channels=True,
    # Decoder
    decoder=DECODER, decoder_hidden_sizes=DECODER_HIDDEN_SIZES, output_size=nn_output_shape, dropout=DROPOUT, 
 )
mshred = mSHRED(args=model_args).to(device)

model_path = Path("models_saved/T3B.pth")
model_path.parent.mkdir(parents=True, exist_ok=True)
model_exists = model_path.exists()

#if not already trained, train the model and save it, otherwise load the model from disk
if not model_exists:
    print("Training mSHRED model...")

    # Use cosine annealing LR schedule
    train_errors, valid_errors = fit(
        mshred, train_dataset, val_dataset,
        optimizer=OPTIMIZER, lr=LEARNING_RATE, num_epochs=NUM_EPOCHS, patience=PATIENCE, verbose=VERBOSE_TRAIN,
        lr_scheduler=LR_SCHEDULER, scheduler_step_size=SCHEDULER_STEP_SIZE,
        masking_type=MASKING_TYPE,
     )

    torch.save(mshred.state_dict(), model_path)

else:
    print("Loading pre-trained mSHRED model...")
    mshred.load_state_dict(torch.load(model_path, map_location=device))
    mshred.eval()

eval_all(manager, mshred, train_dataset, val_dataset, test_dataset, H, U, V, model_name="T3B")

del model_args, model_path, mshred
del model_exists


## (T.3.C): Masking hidden space

In [ ]:
# mSHRED TRAINING
model_args = dict(
    # Encoder
    encoder="transformer_hidden_mask", encoder_depth=ENCODER_DEPTH, input_size=nn_input_shape, n_heads=N_HEADS,
    dim_feedforward=DIM_FEEDFORWARD, hidden_size=HIDDEN_SIZE, input_length=LAGS,
    # Decoder
    decoder=DECODER, decoder_hidden_sizes=DECODER_HIDDEN_SIZES, output_size=nn_output_shape, dropout=DROPOUT, 
 )
mshred = mSHRED(args=model_args).to(device)

model_path = Path("models_saved/T3C.pth")
model_path.parent.mkdir(parents=True, exist_ok=True)
model_exists = model_path.exists()

#if not already trained, train the model and save it, otherwise load the model from disk
if not model_exists:
    print("Training mSHRED model...")

    # Use cosine annealing LR schedule
    train_errors, valid_errors = fit(
        mshred, train_dataset, val_dataset,
        optimizer=OPTIMIZER, lr=LEARNING_RATE, num_epochs=NUM_EPOCHS, patience=PATIENCE, verbose=VERBOSE_TRAIN,
        lr_scheduler=LR_SCHEDULER, scheduler_step_size=SCHEDULER_STEP_SIZE,
        masking_type=MASKING_TYPE,
     )

    torch.save(mshred.state_dict(), model_path)

else:
    print("Loading pre-trained mSHRED model...")
    mshred.load_state_dict(torch.load(model_path, map_location=device))
    mshred.eval()

eval_all(manager, mshred, train_dataset, val_dataset, test_dataset, H, U, V, model_name="T3C")

del model_args, model_path, mshred
del model_exists


## (T.3.D): Masking MHSA

In [ ]:
# mSHRED TRAINING
model_args = dict(
    # Encoder
    encoder="transformer_attn_mask", encoder_depth=ENCODER_DEPTH, input_size=nn_input_shape, n_heads=N_HEADS,
    dim_feedforward=DIM_FEEDFORWARD, hidden_size=HIDDEN_SIZE, input_length=LAGS, double_channels=False,
    # Decoder
    decoder=DECODER, decoder_hidden_sizes=DECODER_HIDDEN_SIZES, output_size=nn_output_shape, dropout=DROPOUT, 
 )
mshred = mSHRED(args=model_args).to(device)

model_path = Path("models_saved/T3D.pth")
model_path.parent.mkdir(parents=True, exist_ok=True)
model_exists = model_path.exists()

#if not already trained, train the model and save it, otherwise load the model from disk
if not model_exists:
    print("Training mSHRED model...")

    # Use cosine annealing LR schedule
    train_errors, valid_errors = fit(
        mshred, train_dataset, val_dataset,
        optimizer=OPTIMIZER, lr=LEARNING_RATE, num_epochs=NUM_EPOCHS, patience=PATIENCE, verbose=VERBOSE_TRAIN,
        lr_scheduler=LR_SCHEDULER, scheduler_step_size=SCHEDULER_STEP_SIZE,
        masking_type=MASKING_TYPE,
     )

    torch.save(mshred.state_dict(), model_path)

else:
    print("Loading pre-trained mSHRED model...")
    mshred.load_state_dict(torch.load(model_path, map_location=device))
    mshred.eval()

eval_all(manager, mshred, train_dataset, val_dataset, test_dataset, H, U, V, model_name="T3D")

del model_args, model_path, mshred
del model_exists


## (T.3.E): Masking x2 channels + MHSA

In [ ]:
# mSHRED TRAINING
model_args = dict(
    # Encoder
    encoder="transformer_attn_mask", encoder_depth=ENCODER_DEPTH, input_size=nn_input_shape, n_heads=N_HEADS,
    dim_feedforward=DIM_FEEDFORWARD, hidden_size=HIDDEN_SIZE, input_length=LAGS, double_channels=True,
    # Decoder
    decoder=DECODER, decoder_hidden_sizes=DECODER_HIDDEN_SIZES, output_size=nn_output_shape, dropout=DROPOUT, 
 )
mshred = mSHRED(args=model_args).to(device)

model_path = Path("models_saved/T3E.pth")
model_path.parent.mkdir(parents=True, exist_ok=True)
model_exists = model_path.exists()

#if not already trained, train the model and save it, otherwise load the model from disk
if not model_exists:
    print("Training mSHRED model...")

    # Use cosine annealing LR schedule
    train_errors, valid_errors = fit(
        mshred, train_dataset, val_dataset,
        optimizer=OPTIMIZER, lr=LEARNING_RATE, num_epochs=NUM_EPOCHS, patience=PATIENCE, verbose=VERBOSE_TRAIN,
        lr_scheduler=LR_SCHEDULER, scheduler_step_size=SCHEDULER_STEP_SIZE,
        masking_type=MASKING_TYPE,
     )

    torch.save(mshred.state_dict(), model_path)

else:
    print("Loading pre-trained mSHRED model...")
    mshred.load_state_dict(torch.load(model_path, map_location=device))
    mshred.eval()

eval_all(manager, mshred, train_dataset, val_dataset, test_dataset, H, U, V, model_name="T3E")

del model_args, model_path, mshred
del model_exists


## (T.3.F): Masking MHSA learned

In [ ]:
# mSHRED TRAINING
model_args = dict(
    # Encoder
    encoder="transformer_learned_mask", encoder_depth=ENCODER_DEPTH, input_size=nn_input_shape, n_heads=N_HEADS,
    dim_feedforward=DIM_FEEDFORWARD, hidden_size=HIDDEN_SIZE, input_length=LAGS,
    # Decoder
    decoder=DECODER, decoder_hidden_sizes=DECODER_HIDDEN_SIZES, output_size=nn_output_shape, dropout=DROPOUT, 
 )
mshred = mSHRED(args=model_args).to(device)

model_path = Path("models_saved/T3F.pth")
model_path.parent.mkdir(parents=True, exist_ok=True)
model_exists = model_path.exists()

#if not already trained, train the model and save it, otherwise load the model from disk
if not model_exists:
    print("Training mSHRED model...")

    # Use cosine annealing LR schedule
    train_errors, valid_errors = fit(
        mshred, train_dataset, val_dataset,
        optimizer=OPTIMIZER, lr=LEARNING_RATE, num_epochs=NUM_EPOCHS, patience=PATIENCE, verbose=VERBOSE_TRAIN,
        lr_scheduler=LR_SCHEDULER, scheduler_step_size=SCHEDULER_STEP_SIZE,
        masking_type=MASKING_TYPE,
     )

    torch.save(mshred.state_dict(), model_path)

else:
    print("Loading pre-trained mSHRED model...")
    mshred.load_state_dict(torch.load(model_path, map_location=device))
    mshred.eval()

eval_all(manager, mshred, train_dataset, val_dataset, test_dataset, H, U, V, model_name="T3F")

del model_args, model_path, mshred
del model_exists


## (T.3.G): Masking x2 channels + MHSA learned

In [ ]:
# mSHRED TRAINING
model_args = dict(
    # Encoder
    encoder="transformer_learned_mask", encoder_depth=ENCODER_DEPTH, input_size=nn_input_shape, n_heads=N_HEADS,
    dim_feedforward=DIM_FEEDFORWARD, hidden_size=HIDDEN_SIZE, input_length=LAGS, double_channels=True,
    # Decoder
    decoder=DECODER, decoder_hidden_sizes=DECODER_HIDDEN_SIZES, output_size=nn_output_shape, dropout=DROPOUT, 
 )
mshred = mSHRED(args=model_args).to(device)

model_path = Path("models_saved/T3G.pth")
model_path.parent.mkdir(parents=True, exist_ok=True)
model_exists = model_path.exists()

#if not already trained, train the model and save it, otherwise load the model from disk
if not model_exists:
    print("Training mSHRED model...")

    # Use cosine annealing LR schedule
    train_errors, valid_errors = fit(
        mshred, train_dataset, val_dataset,
        optimizer=OPTIMIZER, lr=LEARNING_RATE, num_epochs=NUM_EPOCHS, patience=PATIENCE, verbose=VERBOSE_TRAIN,
        lr_scheduler=LR_SCHEDULER, scheduler_step_size=SCHEDULER_STEP_SIZE,
        masking_type=MASKING_TYPE,
     )

    torch.save(mshred.state_dict(), model_path)

else:
    print("Loading pre-trained mSHRED model...")
    mshred.load_state_dict(torch.load(model_path, map_location=device))
    mshred.eval()

eval_all(manager, mshred, train_dataset, val_dataset, test_dataset, H, U, V, model_name="T3G")

del model_args, model_path, mshred
del model_exists
del train_dataset, val_dataset, test_dataset


## Results

In [ ]:
import json
from pathlib import Path

metrics_path = Path("reconstruction_metrics_all_models.json")
with metrics_path.open("r", encoding="utf-8") as json_file:
    metrics_by_model = json.load(json_file)

aggregate_field = "Unify all"
sorted_models = sorted(
    (
        (model_name, model_metrics["test"][aggregate_field]["nmae"])
        for model_name, model_metrics in metrics_by_model.items()
        if model_metrics is not None
    ),
    key=lambda item: item[1],
)

print(f"Models sorted by test NMAE - {aggregate_field}:")
for model_name, test_nmae in sorted_models:
    print(f"{model_name}: {test_nmae:.6f}")
